# End-to-End Tabular Classification

A reproducible practice project: predict a synthetic subscription outcome from numeric and categorical features. This dataset is educational, not evidence of real customer behavior.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix


## Create Mixed Data with Missing Values

The target is sampled probabilistically before values are hidden. Missingness is random in this exercise; real missingness can be informative and requires investigation.

In [ ]:
rng = np.random.default_rng(42)
n = 600
age = rng.integers(18, 75, n).astype(float)
visits = rng.poisson(4, n).astype(float)
channel = rng.choice(["web", "store", "partner"], n)
log_odds = -2 + 0.35 * visits + 0.015 * (age - 40) + 0.7 * (channel == "web")
y = pd.Series(rng.binomial(1, 1 / (1 + np.exp(-log_odds))), name="subscribed")
X = pd.DataFrame({"age": age, "visits": visits, "channel": channel})
for column in X:
    X.loc[rng.random(n) < 0.1, column] = np.nan
X.head()


## Reserve the Test Set

Split before learning imputation, scaling, or encoding parameters. Inspect only training data during development. Stratification preserves approximate class proportions; grouped or time-dependent data would require a different split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Training missing values:\n", X_train.isna().sum())
print("Training class proportions:\n", y_train.value_counts(normalize=True))


## Preprocess Each Column Type

Numeric values use training medians followed by standardization. Categories use the most frequent training category and one-hot encoding. Unknown categories at prediction time become all-zero encoded rows for that feature. ColumnTransformer applies each branch to its named columns; Pipeline keeps all fitted transformations inside each cross-validation fold.

In [ ]:
numeric = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric, ["age", "visits"]),
    ("categorical", categorical, ["channel"]),
])
model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])


## Establish a Baseline and Tune on Training Folds

ROC AUC measures probability ranking across thresholds. Compare with a dummy baseline, then search regularization strength using the same reproducible folds. Fold variability is descriptive, not a confidence interval. The best CV score is used for selection and is not an unbiased final performance estimate.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
baseline = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", DummyClassifier(strategy="prior")),
])
print("Baseline CV ROC AUC:", cross_val_score(
    baseline, X_train, y_train, cv=cv, scoring="roc_auc"
).mean())
search = GridSearchCV(
    model, {"classifier__C": [0.01, 0.1, 1, 10]},
    scoring="roc_auc", cv=cv, refit=True,
)
search.fit(X_train, y_train)
print("Selected parameters:", search.best_params_)
pd.DataFrame(search.cv_results_)[["params", "mean_test_score", "std_test_score"]]


## Evaluate Once on Held-Out Data

The selected pipeline has been refitted on the full training set. Report held-out ROC AUC and classification metrics at the default threshold of 0.5. Any threshold tuning must also use training validation folds, never this test set.

In [ ]:
best_model = search.best_estimator_
probabilities = best_model.predict_proba(X_test)[:, 1]
predictions = best_model.predict(X_test)
print("Held-out ROC AUC:", roc_auc_score(y_test, probabilities))
print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions, zero_division=0))


## Predict New Rows

Pass raw columns to the fitted pipeline. The example includes missing values and a previously unseen channel to exercise the preprocessing contract.

In [ ]:
new_rows = pd.DataFrame({
    "age": [35.0, np.nan],
    "visits": [np.nan, 6.0],
    "channel": ["referral", np.nan],
})
new_probabilities = best_model.predict_proba(new_rows)[:, 1]
assert np.isfinite(new_probabilities).all()
pd.DataFrame({"subscription_probability": new_probabilities})
